# Verification Scheduling in Long-Horizon Agent Workflows
## Publication Figures & Architecture Diagrams (Google Colab Ready)

**Student:** Harsh Sharma  
**Institution:** Indian Institute of Information Technology (IIIT), Allahabad  
**Project:** Implementation-Based Research Assignment  
**Dataset:** `final_exp_20261104_015549_5fcb5a61` (4,680 episodes, 90 DAG clusters)  

This notebook reproduces **all diagrams and figures** from the research assignment directly inside Google Colab:
1. **Figure 0:** Conceptual Architecture Flowchart (Planning Degradation & Control Harness Loop)
2. **Figure 1:** Downstream Taint-Consumption Fraction (TCF) across Horizons ($H=12, 24, 36$)
3. **Figure 2:** Fault Detection Delay across Stages (Early, Middle, Late)
4. **Figure 3:** Pareto Frontier: Verification Overhead vs. Error Propagation
5. **Figure 4:** Error Containment Sensitivity across Injected Fault Types
6. **Figure 5:** Paired Contrast Distribution ($C - B$) across $N=90$ Graph Clusters
7. **Figure 6:** Replay Re-execution Computational Overhead Upon Rollback

In [ ]:
# 1. Setup Dependencies and Styling
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import json
from pathlib import Path

# Academic publication formatting (IEEE/ACM style)
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'lines.linewidth': 1.5,
    'grid.alpha': 0.4,
    'grid.linestyle': '--'
})

COLORS = {
    'final_only': '#7f7f7f',       # Grey (Arm A: Final-Only)
    'fixed_periodic': '#1f77b4',   # Deep Blue (Arm B: Primary Baseline)
    'milestone_based': '#ff7f0e',  # Orange (Arm C: Intervention)
    'every_step': '#2ca02c'        # Green (Arm D: Upper Bound)
}

ARM_LABELS = {
    'final_only': 'Arm A (Final-Only, K=1)',
    'fixed_periodic': 'Arm B (Periodic, K=3)',
    'milestone_based': 'Arm C (Milestone, K=3)',
    'every_step': 'Arm D (Every-Step, K=H)'
}
print('Dependencies loaded and academic plot styling active!')

In [ ]:
# 2. Load Verified Experimental Metrics (Embedded for standalone execution)
DEEP_METRICS = {
  "arm_metrics": {
    "final_only": {
      "n_episodes": 1170,
      "fdc_total": 0.3333333333333333,
      "fdc_contract": 0.3333333333333333,
      "mean_delay": 12.166666666666666,
      "std_delay": 10.308608512828105,
      "mean_tcf": 1.0,
      "std_tcf": 0.0,
      "median_tcf": 1.0,
      "iqr_tcf": 0.0,
      "completion_overall": 0.38461538461538464,
      "completion_clean": 1.0,
      "completion_fault": 0.3333333333333333,
      "false_positive_rate": 0.0,
      "mean_calls": 1.0,
      "std_calls": 0.0,
      "mean_workflow_time_ms": 24.996247093400193,
      "std_workflow_time_ms": 16.390778834384047,
      "mean_verifier_time_ms": 0.10216700800686565,
      "std_verifier_time_ms": 0.4574902765245445,
      "mean_replay_nodes": 7.384615384615385,
      "rollback_rate": 0.3076923076923077,
      "api_cost_usd": 0.0
    },
    "fixed_periodic": {
      "n_episodes": 1170,
      "fdc_total": 0.3333333333333333,
      "fdc_contract": 0.3333333333333333,
      "mean_delay": 5.5,
      "std_delay": 5.358979655927632,
      "mean_tcf": 0.861413660474118,
      "std_tcf": 0.28944387561700335,
      "median_tcf": 1.0,
      "iqr_tcf": 0.0,
      "completion_overall": 0.38461538461538464,
      "completion_clean": 1.0,
      "completion_fault": 0.3333333333333333,
      "false_positive_rate": 0.0,
      "mean_calls": 2.7435897435897436,
      "std_calls": 0.5872008989290627,
      "mean_workflow_time_ms": 23.01294282143418,
      "std_workflow_time_ms": 15.39368611955617,
      "mean_verifier_time_ms": 0.18512828964103237,
      "std_verifier_time_ms": 0.5643757288392675,
      "mean_replay_nodes": 4.512820512820513,
      "rollback_rate": 0.3076923076923077,
      "api_cost_usd": 0.0
    },
    "milestone_based": {
      "n_episodes": 1170,
      "fdc_total": 0.3333333333333333,
      "fdc_contract": 0.3333333333333333,
      "mean_delay": 7.805555555555555,
      "std_delay": 7.032829678288325,
      "mean_tcf": 0.90050257627872,
      "std_tcf": 0.24834250232252772,
      "median_tcf": 1.0,
      "iqr_tcf": 0.0,
      "completion_overall": 0.38461538461538464,
      "completion_clean": 1.0,
      "completion_fault": 0.3333333333333333,
      "false_positive_rate": 0.0,
      "mean_calls": 2.7435897435897436,
      "std_calls": 0.5872008989290627,
      "mean_workflow_time_ms": 23.217198632619716,
      "std_workflow_time_ms": 15.440612468287316,
      "mean_verifier_time_ms": 0.17503384698747507,
      "std_verifier_time_ms": 0.11187071168722983,
      "mean_replay_nodes": 3.572649572649573,
      "rollback_rate": 0.3076923076923077,
      "api_cost_usd": 0.0
    },
    "every_step": {
      "n_episodes": 1170,
      "fdc_total": 0.3333333333333333,
      "fdc_contract": 0.3333333333333333,
      "mean_delay": 2.5,
      "std_delay": 4.981861248989003,
      "mean_tcf": 0.7236328214269392,
      "std_tcf": 0.43475725758841793,
      "median_tcf": 1.0,
      "iqr_tcf": 0.8125,
      "completion_overall": 0.38461538461538464,
      "completion_clean": 1.0,
      "completion_fault": 0.3333333333333333,
      "false_positive_rate": 0.0,
      "mean_calls": 21.025641025641026,
      "std_calls": 11.083678564114045,
      "mean_workflow_time_ms": 28.305408120171254,
      "std_workflow_time_ms": 19.220163083781095,
      "mean_verifier_time_ms": 1.0150630755895256,
      "std_verifier_time_ms": 1.41138066349387,
      "mean_replay_nodes": 3.282051282051282,
      "rollback_rate": 0.3076923076923077,
      "api_cost_usd": 0.0
    }
  },
  "primary_contrast": {
    "n_clusters": 90,
    "primary_metric": "Downstream Taint-Consumption Fraction (TCF)",
    "mean_b": 0.8614136604741179,
    "mean_c": 0.90050257627872,
    "mean_diff_c_minus_b": 0.039088915804602026,
    "p_value_two_sided": 0.0,
    "ci_95": [
      0.03661879314554963,
      0.04141381184477585
    ],
    "cohens_d": 3.3496524223710438,
    "delay_contrast": {
      "mean_b": 5.5,
      "mean_c": 7.805555555555555,
      "mean_diff": 2.3055555555555554,
      "p_value": 0.0,
      "ci_95": [
        1.9444444444444444,
        2.672222222222222
      ],
      "cohens_d": 1.2947174200285112
    },
    "calls_contrast": {
      "mean_b": 2.743589743589744,
      "mean_c": 2.743589743589744,
      "mean_diff": 0.0,
      "p_value": 1.0,
      "ci_95": [
        0.0,
        0.0
      ]
    }
  },
  "by_horizon": {
    "12": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "mean_delay": 5.666666666666667,
        "n_episodes": 360
      },
      "fixed_periodic": {
        "mean_tcf": 0.8494598765432099,
        "std_tcf": 0.31243818429461373,
        "mean_delay": 2.3333333333333335,
        "n_episodes": 360
      },
      "milestone_based": {
        "mean_tcf": 0.8724537037037037,
        "std_tcf": 0.28941008905321963,
        "mean_delay": 2.9166666666666665,
        "n_episodes": 360
      },
      "every_step": {
        "mean_tcf": 0.7180555555555556,
        "std_tcf": 0.43460236649228373,
        "mean_delay": 1.0,
        "n_episodes": 360
      }
    },
    "24": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "mean_delay": 12.166666666666666,
        "n_episodes": 360
      },
      "fixed_periodic": {
        "mean_tcf": 0.8650970489205784,
        "std_tcf": 0.2816491168697497,
        "mean_delay": 5.5,
        "n_episodes": 360
      },
      "milestone_based": {
        "mean_tcf": 0.9098336304218658,
        "std_tcf": 0.22835566674130764,
        "mean_delay": 7.916666666666667,
        "n_episodes": 360
      },
      "every_step": {
        "mean_tcf": 0.7253416518122401,
        "std_tcf": 0.43518828011807403,
        "mean_delay": 2.5,
        "n_episodes": 360
      }
    },
    "36": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "mean_delay": 18.666666666666668,
        "n_episodes": 360
      },
      "fixed_periodic": {
        "mean_tcf": 0.8696840559585657,
        "std_tcf": 0.2731861448121016,
        "mean_delay": 8.666666666666666,
        "n_episodes": 360
      },
      "milestone_based": {
        "mean_tcf": 0.9192203947105908,
        "std_tcf": 0.2196212613929372,
        "mean_delay": 12.583333333333334,
        "n_episodes": 360
      },
      "every_step": {
        "mean_tcf": 0.7275012569130216,
        "std_tcf": 0.4356345231832924,
        "mean_delay": 4.0,
        "n_episodes": 360
      }
    }
  },
  "by_position": {
    "early": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "mean_delay": 22.0,
        "n_episodes": 360
      },
      "fixed_periodic": {
        "mean_tcf": 0.7530582389651016,
        "std_tcf": 0.361528478821346,
        "mean_delay": 8.0,
        "n_episodes": 360
      },
      "milestone_based": {
        "mean_tcf": 0.7743175261557613,
        "std_tcf": 0.36836942468171957,
        "mean_delay": 10.416666666666666,
        "n_episodes": 360
      },
      "every_step": {
        "mean_tcf": 0.6903280542986425,
        "std_tcf": 0.44433224709928126,
        "mean_delay": 2.5,
        "n_episodes": 360
      }
    },
    "middle": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "mean_delay": 11.0,
        "n_episodes": 360
      },
      "fixed_periodic": {
        "mean_tcf": 0.8311827424572523,
        "std_tcf": 0.29857566934709584,
        "mean_delay": 6.2,
        "n_episodes": 360
      },
      "milestone_based": {
        "mean_tcf": 0.9271902026803989,
        "std_tcf": 0.15185849940378315,
        "mean_delay": 9.8,
        "n_episodes": 360
      },
      "every_step": {
        "mean_tcf": 0.7305704099821747,
        "std_tcf": 0.4251863886768467,
        "mean_delay": 4.0,
        "n_episodes": 360
      }
    },
    "late": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "mean_delay": 1.0,
        "n_episodes": 360
      },
      "fixed_periodic": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "mean_delay": 1.0,
        "n_episodes": 360
      },
      "milestone_based": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "mean_delay": 1.0,
        "n_episodes": 360
      },
      "every_step": {
        "mean_tcf": 0.75,
        "std_tcf": 0.4336153642602654,
        "mean_delay": 0.0,
        "n_episodes": 360
      }
    }
  },
  "by_fault_type": {
    "missing_field": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 1.0,
        "mean_delay": 11.333333333333334,
        "n_episodes": 270
      },
      "fixed_periodic": {
        "mean_tcf": 0.49869637140878964,
        "std_tcf": 0.36806395098450945,
        "fdc": 1.0,
        "mean_delay": 3.3333333333333335,
        "n_episodes": 270
      },
      "milestone_based": {
        "mean_tcf": 0.618475865698088,
        "std_tcf": 0.3714434877967135,
        "fdc": 1.0,
        "mean_delay": 5.7407407407407405,
        "n_episodes": 270
      },
      "every_step": {
        "mean_tcf": 0.0,
        "std_tcf": 0.0,
        "fdc": 1.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      }
    },
    "type_corruption": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.3333333333333333,
        "mean_delay": 14.666666666666666,
        "n_episodes": 270
      },
      "fixed_periodic": {
        "mean_tcf": 0.9469582704876822,
        "std_tcf": 0.15040324506551175,
        "fdc": 0.3333333333333333,
        "mean_delay": 12.0,
        "n_episodes": 270
      },
      "milestone_based": {
        "mean_tcf": 0.9835344394167923,
        "std_tcf": 0.052780633808117705,
        "fdc": 0.3333333333333333,
        "mean_delay": 14.0,
        "n_episodes": 270
      },
      "every_step": {
        "mean_tcf": 0.8945312857077563,
        "std_tcf": 0.22345467976231273,
        "fdc": 0.3333333333333333,
        "mean_delay": 10.0,
        "n_episodes": 270
      }
    },
    "numeric_corruption": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      },
      "fixed_periodic": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      },
      "milestone_based": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      },
      "every_step": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      }
    },
    "broken_reference": {
      "final_only": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      },
      "fixed_periodic": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      },
      "milestone_based": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      },
      "every_step": {
        "mean_tcf": 1.0,
        "std_tcf": 0.0,
        "fdc": 0.0,
        "mean_delay": 0.0,
        "n_episodes": 270
      }
    }
  },
  "recovery_stats": {
    "final_only": {
      "total_fault_episodes": 1080,
      "rollbacks_triggered": 360,
      "recovered_passed_acceptance": 360,
      "failed_after_rollback": 0,
      "passed_without_rollback": 0,
      "final_acceptance_rate": 0.3333333333333333
    },
    "fixed_periodic": {
      "total_fault_episodes": 1080,
      "rollbacks_triggered": 360,
      "recovered_passed_acceptance": 360,
      "failed_after_rollback": 0,
      "passed_without_rollback": 0,
      "final_acceptance_rate": 0.3333333333333333
    },
    "milestone_based": {
      "total_fault_episodes": 1080,
      "rollbacks_triggered": 360,
      "recovered_passed_acceptance": 360,
      "failed_after_rollback": 0,
      "passed_without_rollback": 0,
      "final_acceptance_rate": 0.3333333333333333
    },
    "every_step": {
      "total_fault_episodes": 1080,
      "rollbacks_triggered": 360,
      "recovered_passed_acceptance": 360,
      "failed_after_rollback": 0,
      "passed_without_rollback": 0,
      "final_acceptance_rate": 0.3333333333333333
    }
  }
}
CLUSTER_DIFFS = [0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.024074, 0.020833, 0.024074, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.045455, 0.043301, 0.045455, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902, 0.04902, 0.05057, 0.04902]

print('Experimental data successfully loaded into memory!')
print(f'N = {DEEP_METRICS["overall"]["eligible_count"]} eligible episodes across {len(CLUSTER_DIFFS)} graph clusters.')

### Diagram 0: Conceptual View of Planning Degradation and Control Harness

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 8.5), dpi=150)
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')

boxes = [
    ('Research goal', 'Objective and success criteria', 10.5, '#f8f9fa', '#2b2d42'),
    ('Planning', 'Choose next action or branch', 9.1, '#f8f9fa', '#2b2d42'),
    ('Execution', 'Tools, code, and experiments', 7.7, '#f8f9fa', '#2b2d42'),
    ('Observation', 'Outputs and environment state', 6.3, '#f8f9fa', '#2b2d42'),
    ('Checkpoint / recovery', 'Rollback or re-anchor', 4.9, '#f8f9fa', '#2b2d42'),
    ('Verification', 'Tests, assertions, evidence checks', 3.5, '#f8f9fa', '#2b2d42'),
    ('Memory / context', 'Stable state and working context', 2.1, '#f8f9fa', '#2b2d42'),
]

box_w = 4.8
box_h = 0.95
box_x = 2.6

for title, subtitle, y_center, fill_color, text_color in boxes:
    rect = patches.FancyBboxPatch(
        (box_x, y_center - box_h / 2), box_w, box_h,
        boxstyle='round,pad=0.15,rounding_size=0.2',
        facecolor=fill_color, edgecolor='#495057', linewidth=1.4, zorder=3
    )
    ax.add_patch(rect)
    ax.text(box_x + box_w / 2, y_center + 0.12, title, ha='center', va='center', fontsize=11, fontweight='bold', color=text_color, zorder=4)
    ax.text(box_x + box_w / 2, y_center - 0.18, subtitle, ha='center', va='center', fontsize=8.5, color='#6c757d', zorder=4)

for i in range(len(boxes) - 1):
    y_top = boxes[i][2] - box_h / 2
    y_bot = boxes[i+1][2] + box_h / 2
    ax.annotate('', xy=(box_x + box_w / 2, y_bot), xytext=(box_x + box_w / 2, y_top),
                arrowprops=dict(arrowstyle='-|>', color='#495057', lw=1.5, mutation_scale=12), zorder=2)

loop_x = 1.3
y_start = 2.1
y_end = 9.1
ax.plot([box_x - 0.2, loop_x, loop_x, box_x - 0.2], [y_start, y_start, y_end, y_end], color='#1d3557', lw=1.8, zorder=2)
ax.annotate('', xy=(box_x, y_end), xytext=(box_x - 0.2, y_end),
            arrowprops=dict(arrowstyle='-|>', color='#1d3557', lw=1.8, mutation_scale=14), zorder=2)
ax.text(loop_x - 0.22, (y_start + y_end) / 2, 'Feedback loop', ha='center', va='center', rotation=90, fontsize=9.5, fontweight='bold', color='#1d3557')
ax.text(box_x + box_w / 2, 0.9, 'Control preserves progress and enables recovery.', ha='center', va='center', fontsize=9, fontstyle='italic', color='#495057')

plt.tight_layout()
plt.savefig('fig_conceptual_architecture.png', dpi=300)
plt.show()

### Figure 1: Downstream Error Propagation (TCF) by Task Horizon ($H=12, 24, 36$)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
horizons = ['12', '24', '36']
x = np.arange(len(horizons))
width = 0.2
arms = ['final_only', 'fixed_periodic', 'milestone_based', 'every_step']

for i, arm in enumerate(arms):
    means = [DEEP_METRICS['by_horizon'][h][arm]['mean_tcf'] for h in horizons]
    stds = [DEEP_METRICS['by_horizon'][h][arm]['std_tcf'] for h in horizons]
    sems = [s / np.sqrt(360) for s in stds]
    offset = (i - 1.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=3,
           label=ARM_LABELS[arm], color=COLORS[arm], alpha=0.9, edgecolor='black', linewidth=0.8)

ax.set_ylabel('Downstream Taint-Consumption Fraction (TCF)')
ax.set_xlabel('Workflow Horizon ($H$ Nodes)')
ax.set_title('Downstream Error Propagation by Task Horizon')
ax.set_xticks(x)
ax.set_xticklabels([f'H = {h}' for h in horizons])
ax.set_ylim(0.5, 1.05)
ax.grid(axis='y')
ax.legend(frameon=True, facecolor='white', loc='lower right')
plt.tight_layout()
plt.savefig('fig1_tcf_by_horizon.png', dpi=300)
plt.show()

### Figure 2: Fault Detection Latency Across Workflow Stages (Early, Middle, Late)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
positions = ['early', 'middle', 'late']
x = np.arange(len(positions))
width = 0.2
arms = ['final_only', 'fixed_periodic', 'milestone_based', 'every_step']

for i, arm in enumerate(arms):
    delays = [DEEP_METRICS['by_position'][p][arm]['mean_delay'] for p in positions]
    offset = (i - 1.5) * width
    ax.bar(x + offset, delays, width, label=ARM_LABELS[arm],
           color=COLORS[arm], alpha=0.9, edgecolor='black', linewidth=0.8)

ax.set_ylabel('Mean Detection Delay (Execution Steps)')
ax.set_xlabel('Fault Injection Position Bin')
ax.set_title('Fault Detection Latency Across Workflow Stages')
ax.set_xticks(x)
ax.set_xticklabels(['Early (1 to H/3)', 'Middle (H/3 to 2H/3)', 'Late (2H/3 to H)'])
ax.set_ylim(0, 25)
ax.grid(axis='y')
ax.legend(frameon=True, facecolor='white', loc='upper right')
plt.tight_layout()
plt.savefig('fig2_detection_delay_comparison.png', dpi=300)
plt.show()

### Figure 3: Verification Overhead vs. Error Propagation Pareto Frontier

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
arms = ['final_only', 'fixed_periodic', 'milestone_based', 'every_step']
calls = [DEEP_METRICS['arm_metrics'][arm]['mean_calls'] for arm in arms]
tcfs = [DEEP_METRICS['arm_metrics'][arm]['mean_tcf'] for arm in arms]
labels = ['Arm A (Final-Only)', 'Arm B (Periodic)', 'Arm C (Milestone)', 'Arm D (Every-Step)']

for arm, call, tcf, label in zip(arms, calls, tcfs, labels):
    ax.scatter(call, tcf, s=200, color=COLORS[arm], edgecolors='black', linewidth=1.2, zorder=5, label=label)

ax.annotate('Arm A (1.00, 1.000)', (calls[0], tcfs[0]), textcoords='offset points', xytext=(-20, 10), ha='center', fontsize=9)
ax.annotate('Arm B: Periodic\n(2.74, 0.861)', (calls[1], tcfs[1]), textcoords='offset points', xytext=(-55, -22), ha='center', fontsize=9, fontweight='bold', color='#1f77b4')
ax.annotate('Arm C: Milestone\n(2.74, 0.901)', (calls[2], tcfs[2]), textcoords='offset points', xytext=(55, 10), ha='center', fontsize=9, fontweight='bold', color='#d95f02')
ax.annotate('Arm D: Every-Step\n(21.03, 0.724)', (calls[3], tcfs[3]), textcoords='offset points', xytext=(-10, 12), ha='center', fontsize=9)

ax.axvline(x=2.7436, color='purple', linestyle=':', alpha=0.7, label='Matched Budget Parity (K = 2.74)')
ax.set_xlabel('Realized Verification Calls ($K_{\\text{realized}}$)')
ax.set_ylabel('Downstream Taint-Consumption Fraction (TCF)')
ax.set_title('Verification Overhead vs. Error Propagation Trade-off')
ax.set_xlim(0, 24)
ax.set_ylim(0.65, 1.05)
ax.grid(True)
ax.legend(frameon=True, facecolor='white', loc='lower right')
plt.tight_layout()
plt.savefig('fig3_tradeoff_overhead_vs_propagation.png', dpi=300)
plt.show()

### Figure 4: Sensitivity of Downstream TCF Across Fault Categories

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
fault_types = ['missing_field', 'type_corruption', 'numeric_corruption', 'broken_reference']
x = np.arange(len(fault_types))
width = 0.2
arms = ['final_only', 'fixed_periodic', 'milestone_based', 'every_step']

for i, arm in enumerate(arms):
    means = [DEEP_METRICS['by_fault_type'][ft][arm]['mean_tcf'] for ft in fault_types]
    offset = (i - 1.5) * width
    ax.bar(x + offset, means, width, label=ARM_LABELS[arm],
           color=COLORS[arm], alpha=0.9, edgecolor='black', linewidth=0.8)

ax.set_ylabel('Mean Taint-Consumption Fraction (TCF)')
ax.set_xlabel('Injected Fault Category')
ax.set_title('Error Containment Sensitivity by Fault Type')
ax.set_xticks(x)
ax.set_xticklabels(['Missing Field', 'Type Corruption', 'Numeric Out-of-Range', 'Broken Reference'])
ax.set_ylim(0, 1.1)
ax.grid(axis='y')
ax.legend(frameon=True, facecolor='white', loc='lower right')
plt.tight_layout()
plt.savefig('fig4_fault_type_sensitivity.png', dpi=300)
plt.show()

### Figure 5: Paired Difference Distribution (TCF_C - TCF_B) Across N=90 Clusters

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
diffs = CLUSTER_DIFFS
n, bins, patches = ax.hist(diffs, bins=15, color='#ff7f0e', alpha=0.7, edgecolor='black', linewidth=0.8)

mean_diff = np.mean(diffs)
ci_low, ci_high = np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

ax.axvline(0, color='black', linestyle='--', linewidth=1.2, label='Null Effect Line (C = B)')
ax.axvline(mean_diff, color='red', linestyle='-', linewidth=2, label=f'Mean Difference: +{mean_diff:.4f}')
ax.axvspan(ci_low, ci_high, color='red', alpha=0.15, label=f'Empirical 95% Interval: [{ci_low:.4f}, {ci_high:.4f}]')

ax.set_xlabel('Paired Cluster Difference $\\Delta \\text{TCF} = \\text{TCF}_C - \\text{TCF}_B$')
ax.set_ylabel('Number of Graph Clusters ($N=90$)')
ax.set_title('Distribution of Paired Scheduling Contrast ($C - B$)')
ax.grid(axis='y')
ax.legend(frameon=True, facecolor='white', loc='upper left')
plt.tight_layout()
plt.savefig('fig5_paired_contrast_distribution.png', dpi=300)
plt.show()

### Figure 6: Replay Re-execution Computational Overhead Upon Rollback

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
arms = ['final_only', 'fixed_periodic', 'milestone_based', 'every_step']
replays = [DEEP_METRICS['arm_metrics'][arm]['mean_replay_nodes'] for arm in arms]
labels = ['Arm A\nFinal-Only', 'Arm B\nPeriodic', 'Arm C\nMilestone', 'Arm D\nEvery-Step']

bars = ax.bar(labels, replays, color=[COLORS[a] for a in arms], alpha=0.9, edgecolor='black', width=0.55)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, yval + 0.15, f'{yval:.2f}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Mean Replay Nodes Executed per Episode')
ax.set_title('Computational Re-execution Overhead Upon Rollback')
ax.set_ylim(0, 9)
ax.grid(axis='y')
plt.tight_layout()
plt.savefig('fig6_replay_overhead.png', dpi=300)
plt.show()

### Download All Figures as a ZIP File
Run the cell below in Google Colab to download all generated PNG figures in a single `.zip` file.

In [ ]:
import zipfile
import os
try:
    from google.colab import files
    in_colab = True
except ImportError:
    in_colab = False

with zipfile.ZipFile('publication_figures.zip', 'w') as zipf:
    for f in Path('.').glob('fig*.png'):
        zipf.write(f)
        print(f'Added {f} to publication_figures.zip')

if in_colab:
    files.download('publication_figures.zip')
    print('Download started in browser!')
else:
    print('Saved publication_figures.zip locally.')